## 1. Data Pre-processing and Exploration
We load the `co2_daily_mlo.csv` file robustly. To avoid imputing missing data, we will search for the longest continuous period of days without missing records.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# Custom parser to handle the CSV formatting anomalies
data_lines = []
with open('co2_daily_mlo.csv', 'r') as file:
    for line in file:
        line = line.strip()
        if line and line[0].isdigit() and ',' in line:
            parts = line.split()
            for part in parts:
                if ',' in part:
                    cols = part.split(',')
                    if len(cols) == 5:
                        data_lines.append(cols)

df = pd.DataFrame(data_lines, columns=['year', 'month', 'day', 'decimal_date', 'co2'])
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)
df['day'] = df['day'].astype(int)
df['co2'] = df['co2'].astype(float)

df['date'] = pd.to_datetime(df[['year', 'month', 'day']])
df.set_index('date', inplace=True)
df.sort_index(inplace=True)

print(f"Total parsed records: {len(df)}")

In [ ]:
# Handle gaps by resampling and linear interpolation
# First ensure all columns are numeric
df['decimal_date'] = df['decimal_date'].astype(float)
# Resample to daily frequency and interpolate
complete_data = df.resample('D').mean()
complete_data['co2'] = complete_data['co2'].interpolate(method='linear')

# Drop any remaining NaNs at the beginning if any
complete_data.dropna(subset=['co2'], inplace=True)

print(f"Data range after interpolation: {complete_data.index[0].date()} to {complete_data.index[-1].date()}")
print(f"Total days: {len(complete_data)}")

## 2. Train/Test Split
We use the last full year as the test validation data (hold-out) and exactly 8 years (approx 2920 observations) before the test set as the training data.

In [ ]:
end_date = complete_data.index[-1]
test_start = end_date - pd.DateOffset(years=1)
train_end = test_start - pd.DateOffset(days=1)
train_start = train_end - pd.DateOffset(years=8)

train_data = complete_data.loc[train_start:train_end]
test_data = complete_data.loc[test_start:end_date]

print(f"Training set: {train_data.index[0].date()} to {train_data.index[-1].date()} ({len(train_data)} obs)")
print(f"Testing set:  {test_data.index[0].date()} to {test_data.index[-1].date()} ({len(test_data)} obs)")

plt.figure(figsize=(10, 4))
plt.plot(train_data.index, train_data['co2'], label='Train')
plt.plot(test_data.index, test_data['co2'], label='Test (Hold-out)')
plt.title('CO2 Concentration Train/Test Split')
plt.legend()
plt.show()

## 3. Removing Trend and Seasonality
To make the residuals stationary, we model the deterministic components explicitly. 

**Mathematical Expressions:**
- **Trend ($T_t$):** A quadratic function of time: $T_t = \beta_0 + \beta_1 t + \beta_2 t^2$.
- **Seasonality ($S_t$):** A 1-year periodic harmonic (Fourier series): $S_t = \alpha_1 \cos(\omega t) + \alpha_2 \sin(\omega t)$, where $\omega = \frac{2\pi}{365.25}$.
- **Combined Model:** $Y_t = T_t + S_t + X_t$, where $X_t$ are the stochastic residuals we aim to model with an ARMA process.

In [ ]:
t_train = np.arange(len(train_data))
X_trend_train = np.column_stack((t_train, t_train**2))

# 365.25 days in a year
omega = 2 * np.pi / 365.25
X_seasonal_train = np.column_stack((np.cos(omega * t_train), np.sin(omega * t_train)))
X_train = np.hstack((X_trend_train, X_seasonal_train))

model = LinearRegression().fit(X_train, train_data['co2'])
trend_seasonal_fit = model.predict(X_train)
residuals = train_data['co2'] - trend_seasonal_fit

plt.figure(figsize=(10, 4))
plt.plot(train_data.index, residuals)
plt.title('Residuals after Removing Trend and Seasonality')
plt.show()

## 4. Stationarity and ARMA Fit
We test the residuals for stationarity using the Augmented Dickey-Fuller test, then fit an ARMA model.

In [ ]:
adf_res = adfuller(residuals)
print(f"ADF Statistic: {adf_res[0]:.4f}")
print(f"p-value: {adf_res[1]:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(residuals, ax=axes[0], lags=40)
plot_pacf(residuals, ax=axes[1], lags=40)
axes[0].set_title('Residuals ACF (Check for decay)')
axes[1].set_title('Residuals PACF')
plt.show()

if adf_res[1] < 0.05:
    print("Residuals are highly stationary (statistically and visually).")
else:
    print("Warning: Residuals may be non-stationary.")

print("Selecting best ARMA(p, q) model using AIC/BIC grid search...")
best_aic = np.inf
best_order = (0, 0, 0)
results_list = []

for p in range(4):
    for q in range(4):
        try:
            model_fit = sm.tsa.ARIMA(residuals, order=(p, 0, q)).fit()
            results_list.append({'p': p, 'q': q, 'aic': model_fit.aic, 'bic': model_fit.bic})
            if model_fit.aic < best_aic:
                best_aic = model_fit.aic
                best_order = (p, 0, q)
        except:
            continue

results_df = pd.DataFrame(results_list)
print(f"Best order (AIC): {best_order}")

# Fit the best model
arma_model = sm.tsa.ARIMA(residuals, order=best_order).fit()
print(arma_model.summary())

## 5. Diagnostic Checks
We inspect the model diagnostics to ensure the model effectively captured the autocorrelation, resulting in random noise.

In [ ]:
fig = arma_model.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()

lb_test = sm.stats.acorr_ljungbox(arma_model.resid, lags=[10], return_df=True)
print("Ljung-Box test for white noise:\n", lb_test)

## 6. Forecasting
We forecast over the test validation period by recombining the deterministic trends/seasonality with the ARMA forecast.

In [ ]:
t_test = np.arange(len(train_data), len(train_data) + len(test_data))
X_trend_test = np.column_stack((t_test, t_test**2))
X_seasonal_test = np.column_stack((np.cos(omega * t_test), np.sin(omega * t_test)))
X_test = np.hstack((X_trend_test, X_seasonal_test))

deterministic_pred = model.predict(X_test)

# ARMA Forecast with confidence intervals
forecast_res = arma_model.get_forecast(steps=len(test_data))
arma_forecast = forecast_res.predicted_mean
conf_int = forecast_res.conf_int(alpha=0.05)  # 95% CI

final_pred = deterministic_pred + arma_forecast.values
lower_bound = deterministic_pred + conf_int.iloc[:, 0].values
upper_bound = deterministic_pred + conf_int.iloc[:, 1].values

plt.figure(figsize=(12, 6))
plt.plot(train_data.index[-500:], train_data['co2'].iloc[-500:], label='Recent Train')
plt.plot(test_data.index, test_data['co2'], label='Actual Test', color='black', alpha=0.7)
plt.plot(test_data.index, final_pred, label='Forecast (ARMA + Deterministic)', color='red')
plt.fill_between(test_data.index, lower_bound, upper_bound, color='red', alpha=0.15, label='95% Confidence Interval')
plt.title('CO2 Concentration Forecast vs Actual with Confidence Intervals')
plt.ylabel('CO2 (ppm)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

errors = test_data['co2'].values - final_pred
rmse = np.sqrt(np.mean(errors**2))
mae = np.mean(np.abs(errors))
mape = np.mean(np.abs(errors / test_data['co2'].values)) * 100

print(f"Forecast Performance Metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"MAPE: {mape:.4f}%")